In [43]:
import torch
from google.colab import drive

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

drive.mount('/content/drive')

True
Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
from pathlib import Path

!mkdir -p /content/lab8_data
!unzip -q "/content/drive/MyDrive/your_face.zip" -d /content/lab8_data

TRAIN_DIR = Path("/content/lab8_data/your_face")

print(TRAIN_DIR.exists())
print(len(list(TRAIN_DIR.glob("*.jpg"))))

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
replace /content/lab8_data/your_face/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/10.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/11.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/12.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/13.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/14.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/15.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/16.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
replace /content/lab8_data/your_face/17.jpg? [y]es, [n]o, [A]ll, [N]one, 

In [52]:
from pathlib import Path
import shutil

src_dir = Path("/content/lab8_data/your_face")
dst_dir = Path("/content/lab8_kohya/train_data/20_skscuong man")

if dst_dir.exists():
    shutil.rmtree(dst_dir)

dst_dir.mkdir(parents=True, exist_ok=True)

for p in src_dir.glob("*.jpg"):
    shutil.copy(p, dst_dir / p.name)

print("Copied:", len(list(dst_dir.glob("*.jpg"))))

Copied: 27


In [54]:
%cd /content

!rm -rf /content/sd-scripts
!git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts

%cd /content/sd-scripts

!pip install -q --upgrade pip

!pip install -q \
  torch torchvision \
  accelerate==0.25.0 \
  transformers==4.36.2 \
  diffusers==0.25.0 \
  huggingface_hub==0.19.4 \
  safetensors \
  bitsandbytes \
  xformers

!pip install -q -r requirements.txt

/content
Cloning into '/content/sd-scripts'...
remote: Enumerating objects: 11105, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 11105 (delta 133), reused 60 (delta 56), pack-reused 10897 (from 3)
Receiving objects: 100% (11105/11105), 18.04 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (7887/7887), done.
/content/sd-scripts
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.4.1 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-h

In [55]:
from pathlib import Path

model_out_dir = "/content/drive/MyDrive/lab8/output"
model_log_dir = "/content/drive/MyDrive/lab8/logs"

Path(model_out_dir).mkdir(parents=True, exist_ok=True)
Path(model_log_dir).mkdir(parents=True, exist_ok=True)

print("Output:", model_out_dir)
print("Logs:", model_log_dir)
print("Input exists:", Path("/content/lab8_kohya/train_data").exists())

Output: /content/drive/MyDrive/lab8/output
Logs: /content/drive/MyDrive/lab8/logs
Input exists: True


In [ ]:
%cd /content/sd-scripts

!accelerate launch --mixed_precision fp16 --num_cpu_threads_per_process 2 train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/content/lab8_kohya/train_data" \
  --output_dir="/content/drive/MyDrive/lab8/output" \
  --logging_dir="/content/drive/MyDrive/lab8/logs" \
  --output_name="skscuong_lora" \
  --resolution="512,512" \
  --train_batch_size=1 \
  --learning_rate=1e-4 \
  --max_train_steps=1500 \
  --mixed_precision="fp16" \
  --gradient_accumulation_steps=1 \
  --save_every_n_epochs=5 \
  --save_model_as="safetensors" \
  --network_module=networks.lora \
  --network_train_unet_only \
  --network_dim=16 \
  --network_alpha=16 \
  --xformers \
  --use_8bit_adam \
  --enable_bucket \
  --gradient_checkpointing

/content/sd-scripts
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-04-29 15:24:01.048701: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/sd-scripts/library/strategy_base.py:96: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/content/sd-scripts/library/custom_train_functions.py:174: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/content/sd-scripts/library/lpw_stable_diffusion.py:70: SyntaxWarning: invalid escape sequence '\('
  \( - literal character '('
/content/sd-scripts/library/sdxl_lp